<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Day_20_Revision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chunking**

Problem

LLMs cannot process an entire PDF directly.

Example:

500-page PDF

Sending everything to the LLM:

Too many tokens
Expensive
Slow

Solution

Split document into smaller pieces.



In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
text=text = """
Generative Pre-trained Transformers (GPT) are a family of neural network models
that use the transformer architecture to create human-like text. They are trained
on massive amounts of data and can perform tasks like translation, summarization,
and question answering.

Large Language Models (LLMs) have a token limit, meaning they cannot process an
entire 500-page PDF directly. This is why we use chunking to break down the document
into smaller pieces, convert those pieces into embedding vectors, and store them
in a vector database like FAISS for quick retrieval.
"""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_text(text)

In [ ]:
chunks

think of a book instead of searching the entire book we just search for page 10,page 12,page 18

Limitations:
very small chunk:
lose context

very large chunks:
waste tokens

# **Embeddings**

Problem

Computers don't understand text.

Example:

What is GPT?

FAISS cannot search words.

It searches numbers.

SOlution:
COnvert text into vectors

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)

Text

 ↓

Vector

 ↓

[0.23, -0.44, 0.81 ...]

Meaning becomes coordinates in vector space.

Limitation

Bad embedding model:

Bad retrieval

# **FAISS**

Problem

Imagine:

10000 chunks

Checking every chunk manually is impossible.

Solution

Store vectors inside FAISS.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np
index = faiss.IndexFlatL2(384)

index.add(
    np.array(embeddings).astype("float32")
)

In [ ]:
query = "What is GPT?"
query_embedding = embedding_model.encode([query]).astype("float32")

In [ ]:
distance,indicies=index.search(
    query_embedding,
    k=3
)

In [ ]:
print("Indices of matching chunks:", indicies)
print("Distances:", distance)

Intuition

FAISS asks:

Which vectors are closest
to the question vector?

Limitation

FAISS only understands vectors.

No deep understanding.

closest vector doesnt always mean best


# **L2 Distance**

In [ ]:
#used in
faiss.IndexFlatL2()

Intuition

Measure distance between vectors.

Smaller distance:

More similar
Limitation

Magnitude affects results.

# **Normalization**

Problem

Magnitude can mislead similarity.

In [ ]:
faiss.normalize_L2(embeddings)

Intuition

Convert vectors to:

Unit vectors

Only direction matters.

Why?

Semantic meaning is often encoded in direction.